In [ ]:
import numpy as np
import tensorflow as tf
import keras
from keras import layers
import matplotlib.pyplot as plt

(x_train, _), (_, _) = keras.datasets.mnist.load_data()

x_train = x_train.reshape(-1, 28, 28, 1).astype("float32") / 255.0


In [ ]:
# Generator
def build_generator(latent_dim):
    model = keras.Sequential([
        keras.Input(shape=(latent_dim,)),
        layers.Dense(7 * 7 * 128),
        layers.Reshape((7, 7, 128)),
        layers.Conv2DTranspose(128, kernel_size=4, strides=2, padding="same"),
        layers.BatchNormalization(),
        layers.LeakyReLU(alpha=0.2),
        layers.Conv2DTranspose(128, kernel_size=4, strides=2, padding="same"),
        layers.BatchNormalization(),
        layers.LeakyReLU(alpha=0.2),
        layers.Conv2D(1, kernel_size=7, padding="same", activation="sigmoid"),
    ])
    return model

#Discriminator
def build_discriminator():
    model = keras.Sequential([
        keras.Input(shape=(28, 28, 1)),
        layers.Conv2D(64, kernel_size=3, strides=2, padding="same"),
        layers.LeakyReLU(alpha=0.2),
        layers.Conv2D(128, kernel_size=3, strides=2, padding="same"),
        layers.LeakyReLU(alpha=0.2),
        layers.GlobalMaxPooling2D(),
        layers.Dense(1),
    ])
    return model

# Define GAN model
def build_gan(generator, discriminator):
    discriminator.trainable = False
    model = keras.Sequential([generator, discriminator])
    return model


In [ ]:
def train_gan(gan, generator, discriminator, dataset, latent_dim, epochs=10, batch_size=128):
    for epoch in range(epochs):
        for batch in dataset:
            # Generate random latent vectors
            noise = tf.random.normal(shape=(batch_size, latent_dim))
            # Generate fake images
            generated_images = generator(noise)
            # Combine fake and real images
            real_images = batch
            combined_images = tf.concat([generated_images, real_images], axis=0)
            # Labels for generated and real images
            labels = tf.concat(
                [tf.ones((batch_size, 1)), tf.zeros((real_images.shape[0], 1))], axis=0
            )
            # Add random noise to labels
            labels += 0.05 * tf.random.uniform(labels.shape)
            # Train discriminator
            with tf.GradientTape() as tape:
                predictions = discriminator(combined_images)
                d_loss = keras.losses.BinaryCrossentropy(from_logits=True)(labels, predictions)
            grads = tape.gradient(d_loss, discriminator.trainable_weights)
            optimizer.apply_gradients(zip(grads, discriminator.trainable_weights))
            # Generate new random latent vectors
            noise = tf.random.normal(shape=(batch_size, latent_dim))
            # Train generator
            with tf.GradientTape() as tape:
                generated_images = generator(noise)
                predictions = discriminator(generated_images)
                g_loss = keras.losses.BinaryCrossentropy(from_logits=True)(tf.zeros_like(predictions), predictions)
            grads = tape.gradient(g_loss, generator.trainable_weights)
            optimizer.apply_gradients(zip(grads, generator.trainable_weights))
        print(f"Epoch {epoch+1}, Generator Loss: {g_loss}, Discriminator Loss: {d_loss}")



In [ ]:
latent_dimensions = [2, 4, 8, 16, 32, 64]

# Train GANs for different latent dimensions
for latent_dim in latent_dimensions:
    # Build and compile models
    generator = build_generator(latent_dim)
    discriminator = build_discriminator()
    gan = build_gan(generator, discriminator)
    optimizer = keras.optimizers.Adam(learning_rate=0.0002, beta_1=0.5)
    gan.compile(optimizer=optimizer, loss=keras.losses.BinaryCrossentropy(from_logits=True))
    # Reshape and normalize data
    dataset = tf.data.Dataset.from_tensor_slices(x_train).shuffle(1000).batch(128)
    # Train GAN
    train_gan(gan, generator, discriminator, dataset, latent_dim)

    # Generate images from random latent vectors
    num_examples_to_generate = 5
    random_latent_vectors = tf.random.normal(shape=(num_examples_to_generate, latent_dim))
    generated_images = generator(random_latent_vectors)

    # Plot generated images
    fig = plt.figure(figsize=(10, 10))
    for i in range(num_examples_to_generate):
        ax = fig.add_subplot(1, num_examples_to_generate, i + 1)
        ax.imshow(generated_images[i, :, :, 0], cmap="gray")
        ax.axis("off")
    plt.show()

KeyboardInterrupt: 